## Using the trained model to predict in our dataset

### First we import the packages and the dataset

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

import pandas as pd

# Start by importing the test data
test_path = Path('data/raw/dataset_test.csv')
df_test = pd.read_csv(test_path)

Working dir: c:\Users\ladak\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Now we perform the datacleaning

In [2]:
from src.data_cleaning import clean_data
from src.model import prepare_features
import numpy as np

# Clean test data
df_test = clean_data(dataset=df_test, is_train=False, categorize_station=True)
test_ids = df_test["id"].copy()

# ── Lag features ────────────────────────────────────────────────────────────
# The saved model was trained with lag features (bikes_lag_*, roll*, station_hour_mean).
# We compute them on the combined train+test series so that test rows can look
# back into the training period for their lag values.

train_path = Path('data/raw/dataset_train.csv')
df_train_raw = pd.read_csv(train_path)
df_train = clean_data(dataset=df_train_raw, is_train=True, categorize_station=True)

# Align category levels so concat doesn't break
all_stations = sorted(
    set(df_train['station_number'].cat.categories.tolist() +
        df_test['station_number'].cat.categories.tolist())
)
df_train['station_number'] = pd.Categorical(df_train['station_number'], categories=all_stations)
df_test['station_number']  = pd.Categorical(df_test['station_number'],  categories=all_stations)

df_test['bikes'] = np.nan          # placeholder – target is unknown for test rows
df_test['_is_test'] = True
df_train['_is_test'] = False

combined = (
    pd.concat([df_train, df_test], ignore_index=True)
    .sort_values(['station_number', 'datetime'])
    .reset_index(drop=True)
)

grp = combined.groupby('station_number', observed=True)['bikes']
combined['bikes_lag_1']   = grp.shift(1)
combined['bikes_lag_2']   = grp.shift(2)
combined['bikes_lag_4']   = grp.shift(4)
combined['bikes_lag_48']  = grp.shift(48)
combined['bikes_lag_336'] = grp.shift(336)
combined['bikes_roll3h_mean'] = grp.transform(
    lambda x: x.shift(1).rolling(6, min_periods=1).mean()
)
combined['bikes_roll3h_std'] = grp.transform(
    lambda x: x.shift(1).rolling(6, min_periods=2).std().fillna(0)
)

station_hour_mean = (
    combined.groupby(['station_number', 'hour'], observed=True)['bikes']
    .mean()
    .rename('station_hour_mean')
    .reset_index()
)
combined = combined.merge(station_hour_mean, on=['station_number', 'hour'], how='left')

# Keep only test rows; fill any remaining NaN lags (when lag falls into test period)
# with station_hour_mean as a reasonable proxy for the unknown previous bike count.
df_test = combined[combined['_is_test']].copy()

lag_cols = ['bikes_lag_1', 'bikes_lag_2', 'bikes_lag_4',
            'bikes_lag_48', 'bikes_lag_336', 'bikes_roll3h_mean']
for col in lag_cols:
    df_test[col] = df_test[col].fillna(df_test['station_hour_mean'])
df_test['bikes_roll3h_std'] = df_test['bikes_roll3h_std'].fillna(0)

print(f"Test rows: {len(df_test):,}  |  NaN lag values remaining: {df_test[lag_cols].isna().sum().sum()}")


Test rows: 537,445  |  NaN lag values remaining: 0


In [3]:
# Apply the same feature preparation as training
DROP_COLS = ["id", "datetime", "name", "_is_test", "bikes"]
df_test_model = prepare_features(X=df_test, drop_cols=DROP_COLS)

print(f"Features: {df_test_model.shape[1]}  →  {df_test_model.columns.tolist()}")


Features: 21  →  ['station_number', 'lat', 'lng', 'hour', 'dayofweek', 'month', 'is_holiday', 'temperature_2m', 'apparent_temperature', 'precipitation', 'wind_speed_10m', 'cloud_cover', 'relative_humidity_2m', 'bikes_lag_1', 'bikes_lag_2', 'bikes_lag_4', 'bikes_lag_48', 'bikes_lag_336', 'bikes_roll3h_mean', 'bikes_roll3h_std', 'station_hour_mean']


In [4]:
df_test

,id,datetime,station_number,name,lat,lng,bikes,hour,minute,dayofweek,...,relative_humidity_2m,_is_test,bikes_lag_1,bikes_lag_2,bikes_lag_4,bikes_lag_48,bikes_lag_336,bikes_roll3h_mean,bikes_roll3h_std,station_hour_mean
9150,2025-03-13 08:30:00_32000,2025-03-13 08:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,NaN,8,30,3,...,88,True,1.000000,1.000000,0.000000,8.000000,3.000000,0.333333,0.516398,8.241470
9151,2025-03-13 09:00:00_32000,2025-03-13 09:00:00,32000,Julius-Raab-Platz,48.211544,16.382374,NaN,9,0,3,...,86,True,9.586842,1.000000,0.000000,11.000000,6.000000,0.400000,0.547723,9.586842
9152,2025-03-13 09:30:00_32000,2025-03-13 09:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,NaN,9,30,3,...,86,True,9.586842,9.586842,1.000000,12.000000,7.000000,0.500000,0.577350,9.586842
9153,2025-03-13 10:00:00_32000,2025-03-13 10:00:00,32000,Julius-Raab-Platz,48.211544,16.382374,NaN,10,0,3,...,72,True,9.857895,9.857895,1.000000,12.000000,7.000000,0.666667,0.577350,9.857895
9154,2025-03-13 10:30:00_32000,2025-03-13 10:30:00,32000,Julius-Raab-Platz,48.211544,16.382374,NaN,10,30,3,...,72,True,9.857895,9.857895,9.857895,12.000000,9.000000,1.000000,0.000001,9.857895
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2687690,2025-04-29 21:30:00_32283,2025-04-29 21:30:00,32283,eLastenräder- Modecenterstraße / The Marks,48.184929,16.413964,NaN,21,30,1,...,60,True,0.884817,0.884817,0.884817,0.884817,0.884817,0.884817,0.000000,0.884817
2687691,2025-04-29 22:00:00_32283,2025-04-29 22:00:00,32283,eLastenräder- Modecenterstraße / The Marks,48.184929,16.413964,NaN,22,0,1,...,63,True,0.876963,0.876963,0.876963,0.876963,0.876963,0.876963,0.000000,0.876963
2687692,2025-04-29 22:30:00_32283,2025-04-29 22:30:00,32283,eLastenräder- Modecenterstraße / The Marks,48.184929,16.413964,NaN,22,30,1,...,63,True,0.876963,0.876963,0.876963,0.876963,0.876963,0.876963,0.000000,0.876963
2687693,2025-04-29 23:00:00_32283,2025-04-29 23:00:00,32283,eLastenräder- Modecenterstraße / The Marks,48.184929,16.413964,NaN,23,0,1,...,66,True,0.903141,0.903141,0.903141,0.903141,0.903141,0.903141,0.000000,0.903141


### Do the predictions

In [5]:
import joblib
import numpy as np

# Load the trained model
model_path = Path('models/lightgbmv2.joblib')
model = joblib.load(model_path)

# Make predictions
predictions = model.predict(df_test_model)
predictions = np.maximum(0, predictions).round().astype(int)


### Prepare the submission

In [6]:
submission = pd.DataFrame({"id": test_ids, "bikes": predictions})

# and now save the predictions
from pathlib import Path

out_path = Path('reports/predictions.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)